In [ ]:
# -*- coding: utf-8 -*-

# =============================================================================
# WIWB DATA DOWNLOADER
# Haalt rasterdata op via de WIWB API en slaat deze op als GeoTIFF-bestanden.
#
# ⚠️  LET OP — GEBRUIK IN KLEINE BATCHES
#     Dit script haalt per dag één bestand op via de WIWB API.
#     Vraag NIET meer dan ~30 dagen tegelijk op: grote hoeveelheden
#     kunnen de API overbelasten, leiden tot time-outs of geblokkeerde
#     verzoeken. Verwerk lange periodes in meerdere kleine runs.
#
# WIWB Handleiding:
# https://portal.hydronet.com/data/files/Technische%20Instructies%20WIWB%20API.pdf
# =============================================================================


# --- Packages -----------------------------------------------------------------

import os, requests, sys
from pathlib import Path
import datetime, base64
from dotenv import load_dotenv
import zipfile

# voeg de root toe aan het pad (één map omhoog vanuit /scripts)
sys.path.append(str(Path().resolve().parent))

# importeer 
from src.wiwb_authclient import WIWBAuthClient
from src.wiwb_apicaller import build_payload, fetch_raster
from src.wiwb_utils import validate_settings, get_summary

 
# =============================================================================
# ✏️  INSTELLINGEN — pas deze waarden aan voordat je het script uitvoert
# =============================================================================

# Databron: de gewenste dataset uit de WIWB API.
# Zie de WIWB handleiding (kolom Databron ID) voor beschikbare opties.
DATABRON = "LIBV.Bodemvocht.Satelliet.Reanalysis.100m"
# Variabelen: de te downloaden variabelen binnen de gekozen databron.
# Zie de WIWB handleiding (kolom Parameter) voor beschikbare opties.
VARIABELEN = ["Bodemvocht", "Onzekerheid_Bodemvocht"]


# Periode: de start- en einddatum van de gewenste data (formaat: "JJJJ-MM-DD").
# ⚠️  Beperk de periode tot maximaal ~30 dagen per run om overbelasting
#     van de API te voorkomen. Verwerk langere periodes in meerdere runs.
# selecteer startdatum
STARTDATUM = datetime.datetime.strptime("2026-05-11", "%Y-%m-%d")
# selecteer einddatum
EINDDATUM   = datetime.datetime.strptime("2026-05-15", "%Y-%m-%d")


# intialiseer basis folder
DIR_BASE = Path(r"...\LIBV-toepassingsscripts")
# Uitvoermap: de map waar de gedownloade GeoTIFF-bestanden worden opgeslagen
DIR_OUTPUT = DIR_BASE / r"static/output"
# Secrets folder: de map waarin de .env met credentials wordt opgeslagen
DIR_SECRETS = DIR_BASE / r"secrets/.env"


# =============================================================================
# EINDE INSTELLINGEN — pas de code hieronder niet aan
# =============================================================================
if __name__ == "__main__":

    # valideer de instellingen
    validate_settings(
        STARTDATUM=STARTDATUM,
        EINDDATUM=EINDDATUM,
        DIR_SECRETS=DIR_SECRETS
    )
    # laad de API-credentials uit het .env-bestand
    load_dotenv(DIR_SECRETS)

    # WIWB client id
    CLIENT_ID     = os.getenv("CLIENT_ID_WIWB")
    # WIWB client secret
    CLIENT_SECRET = os.getenv("CLIENT_SECRET_WIWB")
    
    if CLIENT_ID is None or CLIENT_SECRET is None:
        raise ValueError(
            "CLIENT_ID_WIWB of CLIENT_SECRET_WIWB niet gevonden in het .env-bestand.\n"
            f"Controleer het bestand in: {DIR_SECRETS}"
        )

    # maak de uitvoermap aan als deze nog niet bestaat
    DIR_OUTPUT.mkdir(parents=True, exist_ok=True)

    # toon configuratie instellingen
    get_summary(
        STARTDATUM=STARTDATUM, 
        EINDDATUM=EINDDATUM, 
        DATABRON=DATABRON, 
        VARIABELEN=VARIABELEN, 
        DIR_OUTPUT=DIR_OUTPUT
    )
    
    # initialiseer WIWB authenticatie
    wiwbauthclient = WIWBAuthClient(CLIENT_ID, CLIENT_SECRET)
    # Haal WIWB token op
    headers = wiwbauthclient.execute()

    # voorbeeld: optioneel extent opgeven 
    #extent = {
    #    "Xll": 100000,
    #    "Yll": 400000,
    #    "Xur": 200000,
    #    "Yur": 500000,
    #    "SpatialReference": {"Epsg": 28992}
    #}

    # maak de payload voor API call
    payload = build_payload(
        STARTDATUM.strftime("%Y%m%d") + "000000",
        EINDDATUM.strftime("%Y%m%d") + "000000",
        DATABRON,
        VARIABELEN,
        #extent
    )

    output_file = DIR_OUTPUT / f"{DATABRON}_{STARTDATUM.date()}_tm_{EINDDATUM.date()}.zip"

    try:
        # fetch de data
        fetch_raster(headers, payload, output_file)

        # pak de zip uit in dezelfde map
        with zipfile.ZipFile(output_file, "r") as z:
            z.extractall(DIR_OUTPUT)
        print(f"Uitgepakt in: {DIR_OUTPUT}")
        
        # verwijder de zip na uitpakken
        output_file.unlink()
        print(f"Zip verwijderd: {output_file.name}")
    except Exception as e:
        print(f"  ✗ Ophalen mislukt: {e}")